# Programming Assignment: Missing Value Audit

Welcome to the **Missing Value Audit** assignment!

Missing data is one of the most pervasive challenges in real-world machine learning. Before you can build any model, you need to understand the **extent** and **patterns** of missingness in your data. This notebook teaches you how to systematically audit missing values across a dataset.

As covered in the theory, missing values in real datasets rarely show up only as `NaN`. Numeric placeholders like `-1`, `999`, or `-9999` are common in legacy systems; string placeholders like `"unknown"` or `"null"` appear in text columns; empty strings slip through CSV parsers. A complete detection step combines `df.isna()` with domain-aware scans for out-of-range or semantically empty values.

**What You Will Do in This Assignment:**

* Count missing values per column using `.isnull().sum()`
* Calculate missing value percentages relative to dataset size
* Identify co-occurrence patterns: which rows are missing multiple columns at once
* Create a comprehensive missing value summary audit report
* Detect columns that exceed a high-missingness threshold

Let's get started!

---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">TIPS FOR SUCCESSFUL GRADING OF YOUR ASSIGNMENT:</h4>

* All cells are frozen except for the ones where you need to submit your solutions or when explicitly mentioned you can interact with it.

* In each exercise cell, look for comments `### START CODE HERE ###` and `### END CODE HERE ###`. These show you where to write the solution code. **Do not add or change any code that is outside these comments**.

* You can add new cells to experiment but these will be omitted by the grader, so don't rely on newly created cells to host your solution code, use the provided places for this.

---

## Table of Contents
- [Imports](#imports)
- [1 - Understanding Missing Values](#1---understanding-missing-values)
    - [1.1 - Types of Missingness (MCAR, MAR, MNAR)](#11---types-of-missingness)
    - [1.2 - Sample Dataset](#12---sample-data)
- [2 - Counting Missing Values](#2---counting-missing-values)
    - **[Exercise 1 - `count_missing_values`](#exercise-1---count_missing_values)**
- [3 - Calculating Missing Percentages](#3---calculating-missing-percentages)
    - **[Exercise 2 - `calculate_missing_percentage`](#exercise-2---calculate_missing_percentage)**
- [4 - Identifying Missing Patterns](#4---identifying-missing-patterns)
    - **[Exercise 3 - `identify_missing_patterns`](#exercise-3---identify_missing_patterns)**
- [5 - Creating a Missing Value Summary](#5---creating-a-missing-value-summary)
    - **[Exercise 4 - `create_missing_summary`](#exercise-4---create_missing_summary)**
- [6 - Detecting High Missingness](#6---detecting-high-missingness)
    - **[Exercise 5 - `detect_high_missingness`](#exercise-5---detect_high_missingness)**
- [7 - Visualizing Missing Data](#7---visualizing-missing-data)

<a name='imports'></a>
## Imports

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
import helper_utils
import unittests

<a name='1---understanding-missing-values'></a>
## 1 - Understanding Missing Values

<a name='11---types-of-missingness'></a>
### 1.1 - Types of Missingness (MCAR, MAR, MNAR)

Before diving into code, it is critical to understand that missing values arise from different mechanisms, each with different implications for how to handle them:

1. **MCAR (Missing Completely At Random)**: The probability of a value being missing is constant and independent of all other variables — both observed and unobserved. Formally: $P(R=0 | X_{obs}, X_{mis}) = P(R=0)$. Example: a sensor randomly fails to record a measurement.

2. **MAR (Missing At Random)**: The probability of missingness depends on *other observed* variables, but not on the missing value itself. Formally: $P(R=0 | X_{obs}, X_{mis}) = P(R=0 | X_{obs})$. Example: younger respondents are less likely to report income, but given age the probability of missing income doesn't depend on the actual income value.

3. **MNAR (Missing Not At Random)**: The probability of missingness depends on the missing value itself. Formally: $P(R=0 | X_{obs}, X_{mis}) = P(R=0 | X_{mis})$. Example: high-income individuals are less likely to report their income *because* it is high.

Understanding the mechanism is the first step in choosing the right handling strategy. MCAR is the only case where dropping incomplete rows introduces no bias. MAR opens the door to model-based imputation. MNAR is the most dangerous and requires domain expertise to address.

<a name='12---sample-data'></a>
### 1.2 - Sample Dataset

We will use a synthetic customer dataset generated with known missingness patterns throughout this assignment. The dataset reproduces the kinds of patterns discussed in the theory:
- **MCAR** missingness in `age` (random 10%)
- **MAR** missingness in `income` (higher for unemployed/students)
- **Structural** missingness in `years_experience` (students cannot have work experience)
- **Random** missingness in `satisfaction_score` (15%)

In [ ]:
# Generate sample data with missing values
df = helper_utils.generate_sample_data_with_missing(n_rows=100, random_state=42)

print(f"Dataset shape: {df.shape}")
print(f"\nFirst 10 rows:")
df.head(10)

In [ ]:
# Get basic info — notice 'Non-Null Count' is less than total rows for some columns
print("Dataset Info:")
df.info()

In [ ]:
# In pandas, NaN is the canonical missing-value token for numeric columns
sample_series = pd.Series([1, 2, np.nan, 4, np.nan])
print(f"Sample series:  {sample_series.values}")
print(f"Is null:        {sample_series.isnull().values}")
print(f"Count of nulls: {sample_series.isnull().sum()}")

<a name='2---counting-missing-values'></a>
## 2 - Counting Missing Values

The first step in any missing value audit is to count how many values are missing in each column. `.isnull()` returns a boolean DataFrame of the same shape where `True` marks each missing cell. Chaining `.sum()` collapses each column into a count.

In [ ]:
# Example: Counting missing values
example_df = pd.DataFrame({
    'A': [1, 2, np.nan, 4],
    'B': [np.nan, np.nan, 3, 4],
    'C': [1, 2, 3, 4]
})
print("Example DataFrame:")
print(example_df)
print("\nMissing values per column:")
print(example_df.isnull().sum())

<a name='exercise-1---count_missing_values'></a>
### **Exercise 1 - `count_missing_values`**

**Your Task:**

Implement `count_missing_values` that returns a `pd.Series` where the index is column names and the values are missing-value counts.

**Requirements:**
- Return a `pd.Series` (not a `pd.DataFrame`)
- Index must be the column names of the input DataFrame
- Values must be non-negative integers

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand if you are stuck)</font></b></summary>

**Step 1:** Use `df.isnull()` to create a boolean mask where `True` = missing

**Step 2:** Chain `.sum()` to count `True` values per column

**Formula:** `missing_counts = df.isnull().sum()`

</details>

In [ ]:
# GRADED FUNCTION: count_missing_values
def count_missing_values(df):
    """
    Count the number of missing values in each column of a DataFrame.

    Args:
        df (pd.DataFrame): Input DataFrame.

    Returns:
        pd.Series: Series with column names as index and missing counts as values.
    """
    ### START CODE HERE ###

    missing_counts = None

    ### END CODE HERE ###

    return missing_counts

In [ ]:
# Try your implementation
missing_counts = count_missing_values(df)
print("Missing value counts per column:")
print(missing_counts)
print(f"\nTotal missing values: {missing_counts.sum()}")

In [ ]:
# Test your code!
unittests.exercise_1(count_missing_values)

<a name='3---calculating-missing-percentages'></a>
## 3 - Calculating Missing Percentages

Raw counts are useful, but percentages give you a better sense of *severity* relative to dataset size. According to the widely-used thresholds from the theory:

| Missing % | Typical approach |
|-----------|------------------|
| < 5%      | Safe to impute with simple methods or drop rows |
| 5% – 20%  | Requires careful method selection; consider pattern analysis |
| 20% – 50% | Advanced imputation or indicator variables recommended |
| > 50%     | Consider dropping the column; imputation risk is high |

In [ ]:
# Example: Converting counts to percentages
example_counts = pd.Series({'A': 25, 'B': 50, 'C': 0}, name='missing_count')
total_rows = 100
percentages = (example_counts / total_rows) * 100
print(f"Missing counts:  {example_counts.values}")
print(f"As percentages:  {percentages.values}")

<a name='exercise-2---calculate_missing_percentage'></a>
### **Exercise 2 - `calculate_missing_percentage`**

**Your Task:**

Implement `calculate_missing_percentage` that calculates the percentage of missing values per column.

**Requirements:**
- Return a `pd.Series` with percentages on the **0–100** scale (not 0–1)
- Values should represent what fraction of rows in that column are missing

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand if you are stuck)</font></b></summary>

**Step 1:** Count missing values using `df.isnull().sum()`

**Step 2:** Divide by `len(df)` to get the fraction

**Step 3:** Multiply by `100` to convert to percentage

**Formula:** `percentage = (df.isnull().sum() / len(df)) * 100`

</details>

In [ ]:
# GRADED FUNCTION: calculate_missing_percentage
def calculate_missing_percentage(df):
    """
    Calculate the percentage of missing values in each column.

    Args:
        df (pd.DataFrame): Input DataFrame.

    Returns:
        pd.Series: Series with column names as index and missing percentages (0-100) as values.
    """
    ### START CODE HERE ###

    missing_percentage = None

    ### END CODE HERE ###

    return missing_percentage

In [ ]:
missing_pct = calculate_missing_percentage(df)
print("Missing value percentages per column:")
print(missing_pct.round(2))

In [ ]:
# Test your code!
unittests.exercise_2(calculate_missing_percentage)

<a name='4---identifying-missing-patterns'></a>
## 4 - Identifying Missing Patterns

Beyond counting, it is valuable to understand **co-occurrence patterns**: which rows are missing multiple columns at once? A row where 5 columns are simultaneously missing suggests a data ingestion failure, not random noise.

The `missingno` matrix plot from the theory shows **horizontal bands of white** when entire rows lose multiple values — a classic sign of a batch failure.

In [ ]:
# Example: Number of missing values per row
example_df = pd.DataFrame({
    'A': [1, np.nan, 3, np.nan],
    'B': [np.nan, np.nan, 3, 4],
    'C': [1, 2, 3, 4]
})
print("DataFrame:")
print(example_df)
print("\nMissing values per row:")
print(example_df.isnull().sum(axis=1))

<a name='exercise-3---identify_missing_patterns'></a>
### **Exercise 3 - `identify_missing_patterns`**

**Your Task:**

Implement `identify_missing_patterns` that returns a DataFrame describing *which* columns are missing in each row that contains at least one missing value.

**Requirements:**
- Return a `pd.DataFrame` with only those rows where at least one value is missing
- Include a `'missing_columns'` column containing the list of column names that are `NaN` for that row
- Include a `'missing_count'` column with the integer count of missing columns for that row

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand if you are stuck)</font></b></summary>

**Step 1:** Find rows with any missing value: `df.isnull().any(axis=1)` (boolean series)

**Step 2:** Get their indices

**Step 3:** For each row index, identify which columns are null:
```python
missing_cols = df.columns[df.loc[idx].isnull()].tolist()
```

**Step 4:** Build a list of dicts, then `pd.DataFrame(patterns).set_index('row_index')`

</details>

In [ ]:
# GRADED FUNCTION: identify_missing_patterns
def identify_missing_patterns(df):
    """
    Identify rows with missing values and describe their missingness pattern.

    Args:
        df (pd.DataFrame): Input DataFrame.

    Returns:
        pd.DataFrame: DataFrame indexed by original row index, with columns:
            - 'missing_columns' (list): column names that are missing in that row
            - 'missing_count' (int): number of missing values in that row
    """
    ### START CODE HERE ###

    # Step 1: Boolean mask — True for rows that have at least one missing value
    rows_with_missing = None

    # Step 2: Get original row indices
    missing_indices = None

    # Step 3: Build pattern list
    patterns = []
    for idx in missing_indices:
        row = df.loc[idx]
        missing_cols = None  # list of column names that are null
        missing_count = None  # integer count
        patterns.append({
            'row_index': idx,
            'missing_columns': missing_cols,
            'missing_count': missing_count
        })

    # Step 4: Create result DataFrame
    result = None

    ### END CODE HERE ###

    return result

In [ ]:
patterns = identify_missing_patterns(df)
print(f"Rows with at least one missing value: {len(patterns)}")
print("\nFirst 10 rows with missing patterns:")
patterns.head(10)

In [ ]:
# Test your code!
unittests.exercise_3(identify_missing_patterns)

<a name='5---creating-a-missing-value-summary'></a>
## 5 - Creating a Missing Value Summary

For a professional missing value audit you combine all metrics into a single table, analogous to the `missing_audit()` function shown in the theory. This makes it easy to triage which columns need attention and what dtype context is available.

In [ ]:
# Example of the target output format (from theory)
demo = pd.DataFrame({'missing_count': [4, 2, 1, 0],
                     'missing_%':     [40.0, 20.0, 10.0, 0.0],
                     'dtype':         ['float64', 'float64', 'float64', 'int64']},
                    index=['credit_score', 'income', 'age', 'customer_id'])
print(demo.to_string())

<a name='exercise-4---create_missing_summary'></a>
### **Exercise 4 - `create_missing_summary`**

**Your Task:**

Implement `create_missing_summary` that creates a comprehensive per-column missing value report sorted by missingness severity.

**Requirements:**
- Return a `pd.DataFrame` with one row per column
- Include columns: `'missing_count'`, `'missing_percentage'`, `'dtype'`
- Sort by `'missing_count'` descending

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand if you are stuck)</font></b></summary>

**Step 1:** Calculate missing counts: `df.isnull().sum()`

**Step 2:** Calculate missing percentages: `(df.isnull().sum() / len(df)) * 100`

**Step 3:** Get data types: `df.dtypes`

**Step 4:** Combine into a DataFrame and sort:
```python
summary = pd.DataFrame({'missing_count': ..., 'missing_percentage': ..., 'dtype': ...})
summary.sort_values('missing_count', ascending=False)
```

</details>

In [ ]:
# GRADED FUNCTION: create_missing_summary
def create_missing_summary(df):
    """
    Create a comprehensive missing value summary for all columns.

    Args:
        df (pd.DataFrame): Input DataFrame.

    Returns:
        pd.DataFrame: DataFrame with columns 'missing_count', 'missing_percentage', 'dtype',
                      sorted by missing_count descending.
    """
    ### START CODE HERE ###

    missing_count = None
    missing_percentage = None
    dtype = None

    summary = None

    ### END CODE HERE ###

    return summary

In [ ]:
summary = create_missing_summary(df)
print("Missing Value Summary:")
summary

In [ ]:
# Test your code!
unittests.exercise_4(create_missing_summary)

<a name='6---detecting-high-missingness'></a>
## 6 - Detecting High Missingness

The theory proposes that columns with **> 50%** missingness are candidates for being dropped rather than imputed — filling more than half a column with inferred values introduces substantial risk. However, context matters: a 55% rate in a million-row dataset is less alarming than the same rate in 500 rows.

In [ ]:
# Example: Filter columns by missingness threshold
example_pct = pd.Series({'income': 22.0, 'age': 5.0, 'score': 55.0, 'id': 0.0})
threshold = 20.0
high_missing = example_pct[example_pct > threshold]
print(f"Columns above {threshold}% missing threshold:")
print(high_missing)

<a name='exercise-5---detect_high_missingness'></a>
### **Exercise 5 - `detect_high_missingness`**

**Your Task:**

Implement `detect_high_missingness` that returns a list of column names whose missing percentage exceeds a given threshold.

**Requirements:**
- Accept a `threshold` parameter (float, default `20.0`)
- Return a **list** of column names (strings) that exceed the threshold
- The list may be empty if no column exceeds the threshold

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand if you are stuck)</font></b></summary>

**Step 1:** Calculate missing percentage per column

**Step 2:** Filter: `missing_pct[missing_pct > threshold]`

**Step 3:** Return `.index.tolist()`

</details>

In [ ]:
# GRADED FUNCTION: detect_high_missingness
def detect_high_missingness(df, threshold=20.0):
    """
    Detect columns whose missing percentage exceeds a given threshold.

    Args:
        df (pd.DataFrame): Input DataFrame.
        threshold (float): Percentage threshold (0-100). Default is 20.0.

    Returns:
        list: Column names with missing percentage greater than the threshold.
    """
    ### START CODE HERE ###

    missing_pct = None
    high_missing_cols = None

    ### END CODE HERE ###

    return high_missing_cols

In [ ]:
high_missing = detect_high_missingness(df, threshold=10.0)
print(f"Columns with > 10% missing: {high_missing}")

high_missing_20 = detect_high_missingness(df, threshold=20.0)
print(f"Columns with > 20% missing: {high_missing_20}")

In [ ]:
# Test your code!
unittests.exercise_5(detect_high_missingness)

<a name='7---visualizing-missing-data'></a>
## 7 - Visualizing Missing Data

The theory covers four `missingno` visualizations:
- **Bar plot** — non-null count per column at a glance
- **Matrix plot** — binary grid showing exactly where values are absent; the right-edge sparkline summarizes per-row completeness
- **Heatmap** — pairwise nullity correlation (−1 to +1); columns always missing together → +1, never together → −1
- **Dendrogram** — hierarchical clustering of columns by missingness similarity

Run the cells below to explore these visualizations on the sample dataset.

In [ ]:
# Seaborn / matplotlib missingness heatmap
helper_utils.plot_missing_heatmap(df)

In [ ]:
# Bar chart — missing count per column
helper_utils.plot_missing_bar(df, show_percentage=True)

In [ ]:
# Matrix plot — binary presence/absence grid
helper_utils.plot_missing_matrix(df)

In [ ]:
# Pairwise missingness correlation
helper_utils.plot_missing_correlation(df)

In [ ]:
# Full textual summary of missing patterns
helper_utils.summarize_missing_patterns(df)